# Arquitecturas Lambda y Kappa + Event-Driven & SLOs

Objetivos:
- Profundizar en Lambda/Kappa con decisiones prácticas.
- Definir SLOs de latencia/frescura y cómo medirlos.
- Diseñar flujo CDC→Kafka→Procesamiento→Silver/Gold.
- Versionado de esquemas de eventos y contratos de datos.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Procesamiento por Lotes (Batch) y en Tiempo Real (Streaming) + Data Mesh](02_procesamiento_batch_streaming_y_data_mesh.ipynb) | Siguiente: [Calidad de Datos y Backfill](04_calidad_y_backfill.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. 1. Lambda vs Kappa (profundización)
2. 2. Flujo Event-Driven con CDC
3. 3. SLOs de Latencia y Frescura
4. 4. Versionado de Esquemas y Contratos
5. 5. Observabilidad y Operación
6. 6. Checklist de Decisiones
7. 7. Ejercicio

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Arquitecturas Lambda y Kappa + Event-Driven & SLOs"] --> C["Notebook siguiente"]
```


## 1. Lambda vs Kappa (profundización)

Lambda: dos pipelines (batch + speed) hacia una capa de serving.
Kappa: único pipeline streaming; batch = reproceso del log.

Criterios de elección:
- Requisitos de latencia
- Complejidad operativa (duplicidad de lógicas)
- Costos y skills del equipo
- Necesidad de re-procesamiento histórico frecuente

In [ ]:
lambda_diagram = '''
[Fuentes] --> [Batch Proc] -----
                         \\-> [Serving/Gold]
[Fuentes] --> [Speed Stream] ---/
'''
kappa_diagram = '''
[Fuentes/Event Log] --> [Stream Proc] --> [Serving/Gold]
(Reprocesos históricos leyendo del log)
'''
print('Lambda:')
print(lambda_diagram)
print('Kappa:')
print(kappa_diagram)


## 2. Flujo Event-Driven con CDC

Ejemplo de flujo:
1) CDC de base OLTP (Debezium) publica eventos a Kafka (topic por tabla).
2) Stream processor (Flink/Spark) enriquece y valida, escribe Silver (parquet).
3) Jobs batch diarios generan Gold (KPIs, data marts).
4) Catálogo/Lineage registra datasets y contratos.

In [ ]:
# Pseudocódigo Spark Structured Streaming
spark_code = '''
stream = (spark
  .readStream
  .format("kafka")
  .option("subscribe", "dbserver1.public.orders")
  .load())

parsed = parse_and_validate(stream.value)  -- esquema + reglas

agg = (parsed
  .withWatermark("event_time", "10 minutes")
  .groupBy(window("event_time", "5 minutes"), "store_id")
  .agg(sum("amount").alias("importe")))

(agg
 .writeStream
 .format("parquet")
 .option("checkpointLocation", "/ckpt/orders")
 .start("/silver/orders_5m"))
'''
print(spark_code)

## 3. SLOs de Latencia y Frescura

Definiciones:
- Latencia de procesamiento (event_time → available_time)
- Frescura (now - last_update_time)
- Disponibilidad (porcentaje de tiempo cumpliendo SLO)

Ejemplo: P95 latencia < 5s para alertas; frescura < 15min para dashboards.

In [ ]:
# Cálculo sencillo de P95 de latencia de eventos
import numpy as np
latencias_ms = np.random.lognormal(mean=1.0, sigma=0.5, size=1000) * 100  # simulado
p95 = np.percentile(latencias_ms, 95)
p99 = np.percentile(latencias_ms, 99)
print('P95 (ms):', round(p95,2), ' | P99 (ms):', round(p99,2))
print('Cumple SLO P95<5000ms?', p95 < 5000)

## 4. Versionado de Esquemas y Contratos

Buenas prácticas:
- Registry para esquemas (Avro/JSON Schema/Protobuf)
- Compatibilidad backward/forward/fully
- Estrategia de evolución: campos opcionales, defaults, deprecación
- Tests de contrato y validación en pipeline CI/CD

In [ ]:
# Ejemplo de JSON Schema (evento pago.autorizado v1)
import json
pago_schema_v1 = {
  "$schema": "http://json-schema.org/draft-07/schema#",
  "title": "pago.autorizado",
  "type": "object",
  "properties": {
    "event_id": {"type": "string"},
    "event_time": {"type": "string", "format": "date-time"},
    "payment_id": {"type": "string"},
    "order_id": {"type": "string"},
    "amount": {"type": "number"},
    "currency": {"type": "string"},
    "customer_id": {"type": "string"}
  },
  "required": ["event_id", "event_time", "payment_id", "amount", "currency"]
}
print(json.dumps(pago_schema_v1, indent=2, ensure_ascii=False))

## 5. Observabilidad y Operación

Métricas clave:
- Lag (producer→consumer), throughput, tasa de error
- Porcentaje de ventanas en SLA, downtime por job
- Alertas: P95/P99 fuera de umbral, falta de mensajes, crecimiento de lag

## 6. Checklist de Decisiones

- [ ] Patrón (Lambda/Kappa) justificado por latencia y complejidad
- [ ] Eventos versionados y contratos documentados
- [ ] SLOs concretos con medición (P95, frescura)
- [ ] CDC y re-procesamiento definidos
- [ ] Observabilidad y alertas configuradas
- [ ] Seguridad (PII, cifrado, ACLs)

## 7. Ejercicio

Diseña para el dominio de pagos:
- Elige Lambda o Kappa y define SLOs (P95 y frescura).
- Propón tópicos y contratos de eventos (v1).
- Especifica pipeline CDC→Kafka→Silver→Gold.
- Define métricas y alertas clave.

In [ ]:
plantilla = '''
# Solución (borrador)
Patrón: [Lambda/Kappa]
SLOs: [P95 latencia, frescura dashboards]
Tópicos y contratos: [lista + esquema mínimo]
Pipeline: [CDC->Kafka->Proc->Silver->Gold]
Observabilidad: [lag, throughput, P95, alertas]
'''
print(plantilla)

---
Referencias:
- Kleppmann, M. Designing Data-Intensive Applications.
- Dehghani, Z. Data Mesh.
- Akidau, T. The World Beyond Batch (Google Dataflow model).

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- Apache Kafka: https://kafka.apache.org/
- Apache Spark Structured Streaming: https://spark.apache.org/structured-streaming/
- Referencias del curso: ../04-recursos/referencias.md
